In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Kaggle bootstrap: clone the exact repo once, then install the checkout.
REPO_URL = "https://github.com/DHYEYPATL/sifess-medical-cv.git"
KAGGLE_WORKING = Path("/kaggle/working")
REPO = KAGGLE_WORKING / "sifess-medical-cv"
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    print(f"Using existing checkout: {REPO}")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO, check=True)
os.chdir(REPO)
print("repo:", Path.cwd())


# SIFESS Day-0: real ChestMNIST ablation run

This notebook runs the four locked SSL ablations on the full ChestMNIST training split by default. It records only metrics produced by the training/probe/OOD code; no example or placeholder numbers are included. The first pass uses ResNet-18, 40 epochs, batch size 32, fp16, four local crops, and seed 42 for a T4-friendly signal run.

If the full split is too slow for the available Kaggle session, set `SUBSET = 20000` in the configuration cell before starting; otherwise leave it as `None`.


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    raise RuntimeError("A Kaggle GPU is required for this Day-0 run; enable a T4 accelerator.")


In [ ]:
from pathlib import Path

ABLATIONS = ["vanilla_dino", "sifess_full", "no_c_gating", "random_frame"]
BACKBONE = "resnet18"
EPOCHS = 40
SUBSET = None  # full ChestMNIST train; change to 20000 only if the full run is too slow
BATCH_SIZE = 32
FP16 = True
N_LOCAL = 4
SEED = 42
NUM_WORKERS = 2
IMAGE_SIZE = 224
OUTPUT_ROOT = Path("outputs/day0")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({"ablations": ABLATIONS, "backbone": BACKBONE, "epochs": EPOCHS, "subset": SUBSET, "output": str(OUTPUT_ROOT)})


In [ ]:
import pandas as pd
from IPython.display import display
from sifess.data.medmnist_loader import build_chestmnist
from sifess.engines.linear_probe import run_probe
from sifess.engines.train_ssl import train

# Resolve the actual train split size rather than assuming a dataset cardinality.
train_ds, _ = build_chestmnist(
    split="train", size=IMAGE_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    multicrop=False, subset=None,
)
TRAIN_SIZE = len(train_ds)
LABEL_FRACS = [0.01, 0.1, 1.0]

def labeled_subset(frac):
    return TRAIN_SIZE if frac == 1.0 else max(1, int(round(TRAIN_SIZE * frac)))

probe_rows = []
ssl_summaries = {}
probe_summaries = {}

for ablation in ABLATIONS:
    print(f"\n=== SSL: {ablation} ===")
    ssl_summary = train({
        "seed": SEED,
        "ablation": ablation,
        "backbone": BACKBONE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "subset": SUBSET,
        "image_size": IMAGE_SIZE,
        "n_global": 2,
        "n_local": N_LOCAL,
        "out_dim": 4096,
        "lambda_eq": 0.5,
        "lambda_orth": 0.01,
        "sigma_rho": 1.5,
        "fp16": FP16,
        "num_workers": NUM_WORKERS,
        "lr": 5e-4,
        "weight_decay": 0.04,
        "teacher_momentum": 0.996,
        "output_dir": str(OUTPUT_ROOT),
    })
    ssl_summaries[ablation] = ssl_summary

    # Each fraction is the number of labeled train examples used by the probe.
    # The engine writes the per-epoch CSV; this table records the final observed value.
    probe_summaries[ablation] = {}
    for label_frac in LABEL_FRACS:
        n_labeled = labeled_subset(label_frac)
        frac_tag = f"{label_frac:g}"
        probe_dir = OUTPUT_ROOT / ablation / f"probe_frac_{frac_tag}"
        print(f"--- linear probe: {ablation}, label_frac={label_frac} ({n_labeled} train examples) ---")
        probe_summary = run_probe({
            "seed": SEED,
            "checkpoint": ssl_summary["checkpoint"],
            "backbone": BACKBONE,
            "epochs": 20,
            "batch_size": BATCH_SIZE,
            "subset": n_labeled,
            "image_size": IMAGE_SIZE,
            "fp16": FP16,
            "num_workers": NUM_WORKERS,
            "lr": 1e-3,
            "output_dir": str(probe_dir),
        })
        probe_summaries[ablation][label_frac] = probe_summary
        probe_rows.append({
            "ablation": ablation,
            "label_frac": label_frac,
            "n_labeled": n_labeled,
            "macro_auroc": probe_summary["macro_auroc"],
            "metrics_csv": probe_summary["metrics_csv"],
        })

probe_table = pd.DataFrame(probe_rows)
probe_table.to_csv(OUTPUT_ROOT / "linear_probe_metrics.csv", index=False)
print("\nWrote:", OUTPUT_ROOT / "linear_probe_metrics.csv")
display(probe_table)


In [ ]:
from sifess.engines.eval_ood import run_ood

ood_summaries = {}
for ablation in ["sifess_full", "vanilla_dino"]:
    print(f"\n=== OOD: {ablation} ===")
    ood_summaries[ablation] = run_ood({
        "seed": SEED,
        "checkpoint": ssl_summaries[ablation]["checkpoint"],
        "probe": str(OUTPUT_ROOT / ablation / "probe_frac_1" / "probe.pt"),
        "backbone": BACKBONE,
        "batch_size": BATCH_SIZE,
        "subset": None,
        "severity": 0.75,
        "image_size": IMAGE_SIZE,
        "num_workers": NUM_WORKERS,
        "n_classes": 14,
        "output_dir": str(OUTPUT_ROOT / ablation / "ood"),
    })
print("OOD evaluations completed:", list(ood_summaries))


In [ ]:
import shutil
from IPython.display import display

print("=== SSL summaries (observed) ===")
display(pd.DataFrame(list(ssl_summaries.values())))
print("=== Linear-probe summaries (observed) ===")
display(pd.read_csv(OUTPUT_ROOT / "linear_probe_metrics.csv"))
print("=== OOD summaries (observed) ===")
for name, summary in ood_summaries.items():
    print(name)
    display(pd.DataFrame(summary["results"]))

# The archive contains only artifacts generated by this run.
zip_path = Path(shutil.make_archive("day0_outputs", "zip", root_dir=OUTPUT_ROOT))
print("Download archive:", zip_path.resolve())
